# Corrected AIPW on the completed binary `causl` grid

This checks the sign of the **control-arm residual** in the ATE influence function and refits only AIPW. It regenerates the same `n=5000` observational training dataset for each seed and instrument strength in the completed Frengression–Frugal Flows grid. An independent `n=400` test dataset is generated for the AIPW evaluation, as in the original binary experiment, but its random seed is now recorded. Frengression and Frugal Flows are **not retrained**. The original AIPW results and the other two original baselines are not silently replaced.

The original `xwshen51/frengression/data_causl/utils.py` adds both arm-specific residual terms to `mu1 - mu0`. For the ATE, the control-arm residual must be **subtracted**. This notebook implements the corrected formula in isolation. Its output CSV is written locally under `examples/benchmark_outputs/` after each seed. It contains no saved experimental outputs.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import time

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'examples' / 'compare_frugal_flows_causl.py').is_file()
             and (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open the frengression_pkg checkout in VS Code.')
sys.path.insert(0, str(ROOT / 'examples'))
if importlib.util.find_spec('sklearn') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn'])
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from compare_frugal_flows_causl import generate_causl
from IPython.display import display

OUTPUT_DIR = ROOT / 'examples' / 'benchmark_outputs'
FULL = OUTPUT_DIR / 'causl_full30_n5000_fr1000_ff1000_m400_mc10000.csv'
AIPW = OUTPUT_DIR / 'causl_aipw_corrected_n5000_test400.csv'
if not FULL.is_file():
    raise FileNotFoundError(f'Completed comparison CSV not found: {FULL}')
full = pd.read_csv(FULL)
keys = full[['strength_instr', 'seed']].drop_duplicates().sort_values(['strength_instr', 'seed'])
assert len(full) == 300 and len(keys) == 150, 'Expected completed 30-seed × 5-strength paired grid'
assert set(full.method) == {'frengression', 'frugal_flows_official'}
print('Using completed grid:', FULL)
print('Corrected AIPW output:', AIPW)


In [ ]:
def aipw_ate_score(a, y, e, mu0, mu1):
    a, y, e, mu0, mu1 = (np.asarray(v, dtype=float).reshape(-1)
                            for v in (a, y, e, mu0, mu1))
    if not all(len(v) == len(a) for v in (y, e, mu0, mu1)):
        raise ValueError('AIPW inputs must have matching lengths')
    if np.any((e <= 0) | (e >= 1)):
        raise ValueError('Propensity estimates must be strictly between zero and one')
    return mu1 - mu0 + a / e * (y - mu1) - (1 - a) / (1 - e) * (y - mu0)

# Algebra check: calculate the two augmented means separately and subtract.
a = np.array([0, 1]); y = np.array([2., 5.]); e = np.array([.5, .5])
mu0 = np.array([1., 1.]); mu1 = np.array([4., 4.])
separate = np.mean(mu1 + a / e * (y - mu1)) - np.mean(
    mu0 + (1 - a) / (1 - e) * (y - mu0))
assert np.isclose(np.mean(aipw_ate_score(a, y, e, mu0, mu1)), separate)
print('AIPW sign check passed')

def fit_corrected_aipw(seed, strength, n=5000, n_test=400):
    x, z, y = generate_causl(n, seed, strength)
    # Independent observational test set, like the original binary.ipynb;
    # its explicit seed lets us reconstruct each estimate later.
    test_seed = seed + 1_000_000
    x_test, z_test, y_test = generate_causl(n_test, test_seed, strength)
    a, outcome = x[:, 0].astype(int), y[:, 0]
    start = time.perf_counter()
    propensity = LogisticRegression(random_state=42).fit(z, a)
    mu0_model = RandomForestRegressor(random_state=42).fit(z[a == 0], outcome[a == 0])
    mu1_model = RandomForestRegressor(random_state=42).fit(z[a == 1], outcome[a == 1])
    e = propensity.predict_proba(z_test)[:, 1]
    mu0, mu1 = mu0_model.predict(z_test), mu1_model.predict(z_test)
    estimate = float(aipw_ate_score(x_test[:, 0], y_test[:, 0], e, mu0, mu1).mean())
    return dict(seed=seed, test_seed=test_seed, strength_instr=strength,
                train_n=n, test_n=n_test, method='aipw_corrected',
                estimated_ate=estimate, signed_error=estimate - 2.0,
                ate_error=abs(estimate - 2.0), seconds=time.perf_counter() - start)


In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
if AIPW.exists():
    saved = pd.read_csv(AIPW)
    if not set(['seed', 'strength_instr', 'train_n', 'test_n', 'method']).issubset(saved.columns):
        raise RuntimeError(f'Unexpected existing file: {AIPW}')
    if not ((saved.train_n == 5000) & (saved.test_n == 400) &
            (saved.method == 'aipw_corrected')).all():
        raise RuntimeError(f'Existing AIPW CSV has different settings: {AIPW}')
    if saved.duplicated(['strength_instr', 'seed']).any():
        raise RuntimeError('Duplicate cached strength/seed pairs')
    rows = saved.to_dict('records')
else:
    rows = []
completed = {(float(row['strength_instr']), int(row['seed'])) for row in rows}
for strength, seed in keys.itertuples(index=False, name=None):
    strength, seed = float(strength), int(seed)
    if (strength, seed) in completed:
        continue
    row = fit_corrected_aipw(seed, strength)
    rows.append(row)
    pd.DataFrame(rows).to_csv(AIPW, index=False)
    print(f'Completed strength={strength:.1f}, seed={seed}, ATE={row["estimated_ate"]:.3f}', flush=True)
results = pd.read_csv(AIPW)
if len(results) != 150:
    raise RuntimeError(f'Expected 150 AIPW results, found {len(results)}')
summary = results.groupby('strength_instr').agg(
    seeds=('seed', 'nunique'),
    ate_bias=('signed_error', 'mean'),
    ate_mae=('ate_error', 'mean'),
    ate_rmse=('signed_error', lambda v: float(np.sqrt(np.mean(np.square(v)))))
)
display(summary.round(4))
print('Local AIPW results:', AIPW)


These corrected AIPW fits use the **same training data** as the completed Frengression–Frugal Flows comparison because `causl` is regenerated deterministically from each recorded seed. The independent test dataset has 400 observations, matching the original evaluation size, but was not saved from the original manuscript run and cannot reproduce its exact random draws. This calculation does not update the original Dragonnet or CausalEGM columns. Do not paste these numbers into the old multi-method table without describing the distinct simulation replications and correcting the old AIPW results.
